In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Célzott kórképek definiálása
TARGET_FINDINGS = [
    "Aortic enlargement",
    "Cardiomegaly",
    "Pleural thickening",
    "Pulmonary fibrosis",
    "Lung Opacity"
]

def prepare_dataframe(csv_path):
    df = pd.read_csv(csv_path)

    # 2. Célkórképek bináris oszlopainak létrehozása (One-Hot kódolás szerűen)
    for finding in TARGET_FINDINGS:
        df[finding] = (df['class_name'] == finding).astype(int)

    # Egyéb kórkép jelző (ami nem No finding és nem is a célzott 5)
    df['Other_finding'] = ((df['class_name'] != 'No finding') &
                           (~df['class_name'].isin(TARGET_FINDINGS))).astype(int)

    # 3. Képszintű aggregálás (ha bármelyik rad_id jelölte, max() miatt 1 lesz)
    grouped = df.groupby('image_id').agg({
        'Aortic enlargement': 'max',
        'Cardiomegaly': 'max',
        'Pleural thickening': 'max',
        'Pulmonary fibrosis': 'max',
        'Lung Opacity': 'max',
        'Other_finding': 'max'
    }).reset_index()

    # 4. "No finding" levezetése és a szűrési szabály alkalmazása
    # Összegezzük az 5 célfindinget
    grouped['target_sum'] = grouped[TARGET_FINDINGS].sum(axis=1)

    # Kiszűrjük azokat, ahol nincs célfinding (target_sum == 0), de van MÁS abnormalitás (Other_finding == 1)
    # A projektterv alapján ezeket töröljük:
    valid_mask = ~((grouped['target_sum'] == 0) & (grouped['Other_finding'] == 1))
    clean_df = grouped[valid_mask].copy()

    # Felesleges oszlopok eldobása
    clean_df = clean_df.drop(columns=['Other_finding', 'target_sum'])

    return clean_df

# 5. image_id alapú Split (Train/Valid/Test)
def split_data(df, test_size=0.15, valid_size=0.15, random_state=42):
    # Fontos: a split egysége az image_id, így a metszet biztosan üres lesz a halmazok között.
    train_val_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state)

    # A validációs méretet a maradék adathoz kell igazítani
    val_ratio = valid_size / (1.0 - test_size)
    train_df, valid_df = train_test_split(train_val_df, test_size=val_ratio, random_state=random_state)

    return train_df, valid_df, test_df

In [3]:
import os
import torch
import pydicom
import cv2
from torch.utils.data import Dataset

def read_dicom_image(file_path, target_size=(512, 512)):
    dicom = pydicom.dcmread(file_path)
    data = dicom.pixel_array

    # VOI LUT (Value of Interest Look-Up Table) alkalmazása, ha van
    if 'WindowCenter' in dicom and 'WindowWidth' in dicom:
        from pydicom.pixel_data_handlers.util import apply_voi_lut
        data = apply_voi_lut(dicom.pixel_array, dicom)

    # MONOCHROME1 kezelése (Invertálás) - Ez oldja meg a negatív felvételek átskálázását
    if dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data

    # Normalizálás [0, 1] tartományba
    data = data - np.min(data)
    data = data / np.max(data)

    # Átalakítás 8-bites (0-255) képpé, ha pl. Albumentations augmentációt használtok
    data = (data * 255).astype(np.uint8)

    # Átméretezés egységes méretre
    data = cv2.resize(data, target_size)

    # 1 csatornás (Grayscale) képből 3 csatornás (RGB) csinálása (sok pre-trained CNN ezt várja)
    data = np.stack([data, data, data], axis=-1)

    return data

class VinBigDataDataset(Dataset):
    def __init__(self, df, image_dir, transforms=None):
        """
        df: A tisztított és szétválasztott pandas DataFrame
        image_dir: A DICOM fájlok mappája
        transforms: Albumentations vagy Torchvision pipeline
        """
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transforms = transforms
        self.labels = self.df[TARGET_FINDINGS].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # image_id alapján a file útvonala
        image_id = self.df.loc[idx, 'image_id']
        file_path = os.path.join(self.image_dir, f"{image_id}.dicom")

        # Kép beolvasása és normalizálása
        image = read_dicom_image(file_path)

        # Augmentációk alkalmazása (csak a train halmazon lesz aktív)
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']
        else:
            # Ha nincs külső transform, kézzel alakítjuk Torch tenzorrá (C, H, W formátum)
            image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Címkék (5 valószínűség) kinyerése
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        return image, label

In [ ]:
clean_df = prepare_dataframe("train.csv")

train_df, valid_df, test_df = split_data(clean_df)

train_ds = VinBigDataDataset(df=train_df, image_dir="./train")
valid_ds = VinBigDataDataset(df=valid_df, image_dir="./train")
test_ds = VinBigDataDataset(df=test_df, image_dir="./train")